<a href="https://colab.research.google.com/github/FodelamineD/NLP_Project/blob/main/NLProjet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gradio as gr
import nltk
from transformers import AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer, util

# ==========================================
# 1. INITIALISATION DES MOTEURS (BACKEND)
# ==========================================
class NLPEngine:
    def __init__(self):
        try: nltk.data.find('tokenizers/punkt_tab')
        except LookupError: nltk.download('punkt_tab')

        # Tokenizers
        self.bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
        self.gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")

        # SENTIMENT (Duel) - CPU forcé pour la stabilité
        self.sent_v1 = pipeline("sentiment-analysis", model="finiteautomata/bertweet-base-sentiment-analysis", device=-1)
        self.sent_v2 = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", device=-1)

        # SIMILARITÉ (Duel)
        self.sim_v1 = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
        self.sim_v2 = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cpu')

        # ZERO-SHOT (Duel)
        self.zero_v1 = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
        self.zero_v2 = pipeline("zero-shot-classification", model="cross-encoder/nli-deberta-v3-small", device=-1)

        # NER (Version Simple & Stable)
        self.ner_pipe = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=-1)

    def process_tokenization(self, text):
        nltk_tokens = nltk.word_tokenize(text)
        hf_tokens = self.bert_tokenizer.tokenize(text)
        gpt_tokens = self.gpt_tokenizer.tokenize(text)
        insight = f"BERT: {len(hf_tokens)} tokens | GPT2: {len(gpt_tokens)} tokens. BERT utilise WordPiece, GPT2 utilise BPE."
        return ", ".join(nltk_tokens), len(nltk_tokens), ", ".join(hf_tokens), len(hf_tokens), ", ".join(gpt_tokens), len(gpt_tokens), insight

    def process_sentiment_dual(self, text):
        res1 = self.sent_v1(text)[0]
        res2 = self.sent_v2(text)[0]
        return {res1['label']: res1['score']}, {res2['label']: res2['score']}

    def process_similarity_dual(self, query, docs_string):
        docs = [d.strip() for d in docs_string.split("\n") if d.strip()]
        if not docs: return "Erreur", "Erreur"
        scores1 = util.cos_sim(self.sim_v1.encode(query), self.sim_v1.encode(docs))[0]
        scores2 = util.cos_sim(self.sim_v2.encode(query), self.sim_v2.encode(docs))[0]
        res1 = "\n".join([f"[{s*100:.1f}%] {d}" for d, s in sorted(zip(docs, scores1), key=lambda x: x[1], reverse=True)])
        res2 = "\n".join([f"[{s*100:.1f}%] {d}" for d, s in sorted(zip(docs, scores2), key=lambda x: x[1], reverse=True)])
        return res1, res2

    def process_zeroshot_dual(self, text, labels_raw):
        labels = [l.strip() for l in labels_raw.split(",") if l.strip()]
        if not labels: return {"Erreur": 1.0}, {"Erreur": 1.0}
        res1 = self.zero_v1(text, labels)
        res2 = self.zero_v2(text, labels)
        return {l: s for l, s in zip(res1['labels'], res1['scores'])}, {l: s for l, s in zip(res2['labels'], res2['scores'])}

    def process_ner(self, text, filters):
        entities = self.ner_pipe(text)
        output, last_end = [], 0
        for ent in entities:
            if ent['start'] > last_end: output.append((text[last_end:ent['start']], None))
            label = ent['entity_group']
            output.append((text[ent['start']:ent['end']], label if label in filters else None))
            last_end = ent['end']
        if last_end < len(text): output.append((text[last_end:], None))
        return output

engine = NLPEngine()

# ==========================================
# 2. DESIGN DE L'INTERFACE (VIOLET)
# ==========================================
custom_theme = gr.Theme.from_hub("freddyaboulton/dracula_revamped").set(
    button_primary_background_fill="#8b5cf6",
    button_primary_background_fill_hover="#a78bfa",
    button_primary_text_color="#ffffff",
    block_border_color="#8b5cf6",
    input_background_fill_focus="#2d333b"
)

with gr.Blocks(title="NLP Master Toolkit", theme=custom_theme) as demo:
    gr.Markdown("# 🛠️ NLP Intelligence Hub")

    with gr.Tabs():
        with gr.Tab("1. Tokenisation"):
            txt_in = gr.Textbox(label="Texte", value="Gradio is amazing for NLP!")
            btn = gr.Button("Analyser", variant="primary")
            with gr.Row():
                out_nltk = gr.Textbox(label="NLTK (Mots)")
                out_hf = gr.Textbox(label="BERT (Subwords)")
                out_gpt = gr.Textbox(label="GPT2 (BPE)")
            insight = gr.Textbox(label="Benchmark Insight")
            btn.click(engine.process_tokenization, txt_in, [out_nltk, gr.Number(visible=False), out_hf, gr.Number(visible=False), out_gpt, gr.Number(visible=False), insight])

        with gr.Tab("2. Sentiment"):
            txt_in = gr.Textbox(label="Saisir une opinion", value="This project is okay, not great but not bad.")
            btn = gr.Button("Comparer", variant="primary")
            with gr.Row():
                out1 = gr.Label(label="BERTweet (Twitter)")
                out2 = gr.Label(label="RoBERTa (Nuancé)")
            btn.click(engine.process_sentiment_dual, txt_in, [out1, out2])

        with gr.Tab("3. Similarité"):
            with gr.Row():
                q = gr.Textbox(label="Requête", value="Le chat dort.")
                db = gr.Textbox(label="Documents", lines=5, value="Une félidée se repose.\nLe chien court.")
            btn = gr.Button("Calculer", variant="primary")
            with gr.Row():
                out1 = gr.Textbox(label="Anglais Uniquement")
                out2 = gr.Textbox(label="Multilingue")
            btn.click(engine.process_similarity_dual, [q, db], [out1, out2])

        with gr.Tab("4. Zero-Shot"):
            txt = gr.Textbox(label="Texte", value="The team won the cup.")
            labs = gr.Textbox(label="Catégories", value="Sport, Politics")
            btn = gr.Button("Classer", variant="primary")
            with gr.Row():
                out1 = gr.Label(label="BART-Large")
                out2 = gr.Label(label="DeBERTa-v3")
            btn.click(engine.process_zeroshot_dual, [txt, labs], [out1, out2])

        with gr.Tab("5. NER & Entités"):
            txt = gr.Textbox(label="Texte", value="Elon Musk a visité Paris.")
            checks = gr.CheckboxGroup(["PER", "ORG", "LOC", "MISC"], value=["PER", "LOC"], label="Filtres")
            btn = gr.Button("Extraire", variant="primary")
            out = gr.HighlightedText(label="Entités", color_map={"PER": "blue", "LOC": "red"})
            btn.click(engine.process_ner, [txt, checks], out)

if __name__ == "__main__":
    demo.launch()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: finiteautomata/bertweet-base-sentiment-analysis
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_790/1934246910.py:85: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="NLP Master Toolkit", theme=custom_theme) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f966acb6fdb48a2344.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
